In [1]:
%pip install --quiet python-dotenv pydantic-ai pandas ffmpeg-python

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/04-video-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Load API keys: Colab secrets or local .env
try:
    from google.colab import userdata
    os.environ.setdefault("GOOGLE_API_KEY", userdata.get("GOOGLE_API_KEY"))
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


Note: you may need to restart the kernel to use updated packages.


# Video

What can you do with a video? It's images + audio + time. Decompose it, then use the tools you already have.

## Download


**`video/download.py`** — Download a video from a URL using yt-dlp


In [2]:
import subprocess
from pathlib import Path

DATA = Path("data")
URL = "https://www.youtube.com/watch?v=rDXubdQdJYs"

DATA.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "yt-dlp",
    "-o", str(DATA / "%(title)s [%(id)s].%(ext)s"),
    "-f", "best",
    "--no-overwrites",
    URL,
])


         To let yt-dlp download and merge the best available formats, simply do not pass any format selection.
         If you know what you are doing and want only the best pre-merged format, use "-f b" instead to suppress this warning


[youtube] Extracting URL: https://www.youtube.com/watch?v=rDXubdQdJYs
[youtube] rDXubdQdJYs: Downloading webpage


[youtube] rDXubdQdJYs: Downloading android vr player API JSON
[youtube] rDXubdQdJYs: Downloading web safari player API JSON


[youtube] rDXubdQdJYs: Downloading player 00c52fa0-tv
[youtube] [jsc:deno] Solving JS challenges using deno


[info] rDXubdQdJYs: Downloading 1 format(s): 18
[download] data/The CNN Presidential debate in 60 seconds [rDXubdQdJYs].mp4 has already been downloaded
[download] 100% of    2.63MiB


CompletedProcess(args=['yt-dlp', '-o', 'data/%(title)s [%(id)s].%(ext)s', '-f', 'best', '--no-overwrites', 'https://www.youtube.com/watch?v=rDXubdQdJYs'], returncode=0)

## Extract frames

Decompose into one image per second. Now you have images — use the image tools.


**`video/frames.py`** — Extract frames from a video at 1 fps using ffmpeg-python


In [3]:
import ffmpeg
from pathlib import Path

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
OUTPUT = Path("outputs") / "frames"
OUTPUT.mkdir(parents=True, exist_ok=True)

(
    ffmpeg
    .input(str(VIDEO))
    .filter("fps", fps=1)
    .output(str(OUTPUT / "frame_%04d.jpg"), **{"qscale:v": 2})
    .overwrite_output()
    .run(quiet=True)
)

frames = sorted(OUTPUT.glob("frame_*.jpg"))
print(f"Extracted {len(frames)} frames to {OUTPUT}")


Extracted 59 frames to outputs/frames


## Extract audio

Extract the audio track. Now you have audio — use the audio tools.


**`video/audio.py`** — Extract audio track from a video file using ffmpeg-python


In [4]:
import ffmpeg
from pathlib import Path

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
OUTPUT = Path("outputs")
OUTPUT.mkdir(parents=True, exist_ok=True)

(
    ffmpeg
    .input(str(VIDEO))
    .output(str(OUTPUT / "rDXubdQdJYs.wav"), ar=16000, ac=1, acodec="pcm_s16le", vn=None)
    .overwrite_output()
    .run(quiet=True)
)

print(f"Audio saved to {OUTPUT / 'rDXubdQdJYs.wav'}")


Audio saved to outputs/rDXubdQdJYs.wav


This is the pipeline behind real investigations. Documented examined hundreds of TikTok videos in French and Wolof: download, extract audio, transcribe with Whisper. Público processed 7,616 TikTok health videos the same way, then used GPT-4o to extract verifiable claims from the transcripts. Video → audio → text → structured data.

## The wrong way

Ask Gemini a question about the video. Get a confident answer with no evidence.


**`video/vibe-answer.py`** — The WRONG way: ask Gemini "who got more screen time?" -- confident answer, no evidence


In [5]:
import time
from pathlib import Path

from google import genai

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
MODEL = "gemini-2.5-flash"

client = genai.Client()
video_file = client.files.upload(file=str(VIDEO), config={"display_name": VIDEO.name})

while video_file.state.name == "PROCESSING":
    time.sleep(5)
    video_file = client.files.get(name=video_file.name)

response = client.models.generate_content(
    model=MODEL,
    contents=[
        "Who got more screen time in this debate video? "
        "Give me a breakdown of approximately how much time each person was on screen.",
        video_file,
    ],
)

print(response.text)


Based on the video provided:

*   **Joe Biden** had approximately **26 seconds** of screen time.
*   **Donald Trump** had approximately **24 seconds** of screen time.

Therefore, **Joe Biden** had slightly more screen time in this debate video.


You can't fact-check this. You can't show your editor the work. You can't catch errors. It's a vibe.

## The right way

Classify each frame with an LLM. Produce an auditable CSV — every row links to a frame you can check.


**`video/decompose-classify.py`** — The RIGHT way: classify each frame with Pydantic AI, produce an auditable CSV


In [6]:
import csv
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
FRAMES_DIR = DATA / "debate"
OUTPUT = Path("outputs") / "frame_classifications.csv"

class FrameClassification(BaseModel):
    subject: str
    confidence: float
    speaking: bool
    description: str

agent = Agent(
    "openai:gpt-4o-mini",
    output_type=FrameClassification,
    system_prompt="Classify frames from a political debate. Identify who is on screen, confidence 0-1, whether they are speaking.",
)

frames = sorted(f for f in FRAMES_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "subject", "confidence", "speaking", "description"])
    for i, path in enumerate(frames, 1):
        c = agent.run_sync([f"Frame {i}", BinaryContent(data=path.read_bytes(), media_type="image/jpeg")]).output
        w.writerow([i, c.subject, f"{c.confidence:.2f}", c.speaking, c.description])
        print(f"{path.name}: {c.subject} ({c.confidence:.2f})")
print(f"\nSaved to {OUTPUT}")


frame-001.jpg: Joe Biden (0.70)


frame-002.jpg: Biden (0.90)


frame-003.jpg: Donald Trump (0.90)


frame-004.jpg: Joe Biden (0.90)


frame-005.jpg: Donald Trump (1.00)


...

frame-026.jpg: Unknown (0.80)


frame-027.jpg: Donald Trump (0.90)


frame-028.jpg: Joe Biden (0.95)


frame-029.jpg: Joe Biden (0.90)


frame-030.jpg: Unknown (0.80)

Saved to outputs/frame_classifications.csv


**Compare the two.** Same question, same video. One gives you a number. The other gives you a spreadsheet with an audit trail. Even if the vibe answer is right, you can't verify it. That's the point.
